<a href="https://colab.research.google.com/github/aryan802/kaggle_sql/blob/main/intro_to_sql_kaggle_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from google.cloud import bigquery
# python package to use bigquery

# 1st step is to create a Client object
# Client object will play a central role in retrieving  information from BigQuery datasets


In [ ]:
client = bigquery.Client()

Using Kaggle's public dataset BigQuery integration.


# constructing a reference to the dataset with the dataset() method.
# use the get_dataset() method, along with the reference we just constructed, to fetch the dataset.


In [ ]:
# Construct a reference to the "hacker_news" dataset
dataset_ref = client.dataset("hacker_news", project= "bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

Every dataset is just a collection of tables.

We use the list_tables() method to list the tables in the dataset.

In [ ]:
# List all tables in "hacker_news" dataset
tables = list(client.list_tables(dataset))

# Print names of all tables in the dataset
for table in tables:
    print(table.table_id)

full


Similar to how we fetched a dataset, we can fetch a table. In the code cell below, we fetch the full table in the hacker_news dataset.

In [ ]:
# Construct a reference to the "full" table
table_ref = dataset_ref.table("full")

# API request - fetch the table
table = client.get_table(table_ref)

Table schema

The structure of a table is called its schema. We need to understand a table's schema to effectively pull out the data we want.

In this example, we'll investigate the full table that we fetched above.

In [ ]:
# Print information on all the columns in the full table in the "hacker_news" dataset
table.schema

[SchemaField('title', 'STRING', 'NULLABLE', None, 'Story title', (), None, None),
 SchemaField('url', 'STRING', 'NULLABLE', None, 'Story url', (), None, None),
 SchemaField('text', 'STRING', 'NULLABLE', None, 'Story or comment text', (), None, None),
 SchemaField('dead', 'BOOLEAN', 'NULLABLE', None, 'Is dead?', (), None, None),
 SchemaField('by', 'STRING', 'NULLABLE', None, "The username of the item's author.", (), None, None),
 SchemaField('score', 'INTEGER', 'NULLABLE', None, 'Story score', (), None, None),
 SchemaField('time', 'INTEGER', 'NULLABLE', None, 'Unix time', (), None, None),
 SchemaField('timestamp', 'TIMESTAMP', 'NULLABLE', None, 'Timestamp for the unix time', (), None, None),
 SchemaField('type', 'STRING', 'NULLABLE', None, 'type of details (comment comment_ranking poll story job pollopt)', (), None, None),
 SchemaField('id', 'INTEGER', 'NULLABLE', None, "The item's unique id.", (), None, None),
 SchemaField('parent', 'INTEGER', 'NULLABLE', None, 'Parent comment ID', (),

Each SchemaField tells us about a specific column (which we also refer to as a field). In order, the information is:

The name of the column
The field type (or datatype) in the column
The mode of the column ('NULLABLE' means that a column allows NULL values, and is the default)
A description of the data in that column
The first field has the SchemaField:

SchemaField('by', 'string', 'NULLABLE', "The username of the item's author.",())

This tells us:

the field (or column) is called by,
the data in this field is strings,
NULL values are allowed, and
it contains the usernames corresponding to each item's author.

We can use the list_rows() method to check just the first five lines of of the full table to make sure this is right. (Sometimes databases have outdated descriptions, so it's good to check.) This returns a BigQuery RowIterator object that can quickly be converted to a pandas DataFrame with the to_dataframe() method.

In [ ]:
# Preview the first five lines of the "full" table
client.list_rows(table, max_results = 5).to_dataframe()

,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,"The loss of OnePlus in the US will sting but, ...",https://9to5google.com/2026/03/29/oneplus-us-w...,None,<NA>,neogodless,4,1774881723,2026-03-30 14:42:03+00:00,story,47575001,<NA>,0,<NA>,<NA>
1,IHP Haskell Framework v1.5 has been released,https://github.com/digitallyinduced/ihp/releas...,None,<NA>,_query,6,1774881814,2026-03-30 14:43:34+00:00,story,47575018,<NA>,0,<NA>,<NA>
2,PicoUnits: Lightweight units and DSL for scien...,https://github.com/wgbowley/PicoUnits,None,<NA>,wgbowley,2,1774881879,2026-03-30 14:44:39+00:00,story,47575030,<NA>,1,<NA>,<NA>
3,"In Case of Emergency, Make Burrito Bison 3 (2017)",https://juicybeast.com/2017/08/03/in-case-of-e...,None,<NA>,amarcheschi,27,1774881935,2026-03-30 14:45:35+00:00,story,47575039,<NA>,8,<NA>,<NA>
4,Show HN: Building a GPT from scratch: What I l...,https://twitter.com/pirosb3/status/20383745029...,None,<NA>,pirosb3,2,1774881943,2026-03-30 14:45:43+00:00,story,47575041,<NA>,0,<NA>,<NA>


In [ ]:
# Preview the first five entries in the "by" column of the "full" table
client.list_rows(table, selected_fields=table.schema[:1], max_results=5).to_dataframe()

,title
0,"The loss of OnePlus in the US will sting but, ..."
1,IHP Haskell Framework v1.5 has been released
2,PicoUnits: Lightweight units and DSL for scien...
3,"In Case of Emergency, Make Burrito Bison 3 (2017)"
4,Show HN: Building a GPT from scratch: What I l...
